In [0]:
%run "../Includes/configurations"

In [0]:
race_results=spark.read.parquet(f"{presentation_folder_path}/race_results")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *


In [0]:
driver_standing=race_results.groupby("year", "driver_name", "driver_nationality", "team").agg(sum("points").alias("total_points"))

In [0]:
driver_standings=driver_standing.filter("year=2020")

In [0]:
driver_rnk_spe=Window.partitionBy("year").orderBy(desc("total_points"), desc("team"))
final_df=driver_standings.withColumn("rank", rank().over(driver_rnk_spe))

In [0]:
final_df.display()

year,driver_name,driver_nationality,team,total_points,rank
2020,Lewis Hamilton,British,Mercedes,347.0,1
2020,Valtteri Bottas,Finnish,Mercedes,223.0,2
2020,Max Verstappen,Dutch,Red Bull,214.0,3
2020,Sergio Pérez,Mexican,Racing Point,125.0,4
2020,Daniel Ricciardo,Australian,Renault,119.0,5
2020,Alexander Albon,Thai,Red Bull,105.0,6
2020,Carlos Sainz,Spanish,McLaren,105.0,7
2020,Charles Leclerc,Monegasque,Ferrari,98.0,8
2020,Lando Norris,British,McLaren,97.0,9
2020,Lance Stroll,Canadian,Racing Point,75.0,10


In [0]:
final_df.write.mode("overwrite").parquet(f"{presentation_folder_path}/driver_standing")